## Configuration

Set your inference parameters here:

In [1]:
# Inference configuration - modify these values as needed
config = {
    'dataset': 'aae2/adus',
    'model_name': 'unsloth/gemma-3-4b-it',
    'epochs': 5,
    'batch_size': 8,
    'gradient_steps': 1,
    'lora': True,
    'lora_r': 16,
    'lora_modules': 'q_proj,k_proj,v_proj,gate_proj,up_proj,down_proj',
    'bit4': False,
    'bit8': False,
    'load_pretrained': False,
    'all_snapshots': True,  # Set to True to run inference on all epoch snapshots
    'remove_system_message': False
}

## Import Libraries

In [2]:
import json
import os
import re
from math import floor
import gc

import torch
from tqdm import tqdm
from unsloth import FastLanguageModel

import chat_templates
from settings import Settings

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Helper Functions

In [3]:
def convert_to_chat_json(text, should_remove_system_role=False, is_gemma_3=False):
    if not should_remove_system_role:
        messages =  json.loads(text)
    else:
        chat = json.loads(text)
        system_message = ''
        user_message = ''
        assistant_message = ''
        for turn in chat:
            if turn['role'] == 'system':
                system_message = turn['content']
            if turn['role'] == 'user':
                user_message = turn['content']
            if turn['role'] == 'assistant':
                assistant_message = turn['content']
        messages = [{'role': 'user', 'content': system_message + '\n' + user_message}, 
                {'role': 'assistant', 'content': assistant_message}]
    if is_gemma_3:
        messages = [ {'role': m['role'], 'content': [{'text': m['content'], 'type': 'text'}] } for m in messages ]

        return messages

def get_gpu_memory_usage():
    if torch.cuda.is_available():
        gpu_memory = torch.cuda.memory_allocated() / (1024**2)  # in MB
        return gpu_memory
    else:
        return "GPU is not available."

## Setup Output Directory and Settings

In [4]:
# Build output directory name
use_lora = config['lora']
output_extra_detail = ''
output_extra_detail += f"lora-r{config['lora_r']}-{''.join([r[0] for r in config['lora_modules'].split(',')])}" if use_lora else ""
output_extra_detail += f"-bs{config['batch_size']}"
output_extra_detail += f"-ac{config['gradient_steps']}"
output_extra_detail += f"-e{config['epochs']}"
output_extra_detail += "-q4" if config['bit4'] else ""
output_extra_detail += "-q8" if config['bit8'] else ""
output_extra_detail += "-fp" if (not (config['bit4'] and config['bit8'])) else ""

settings = Settings(
    dataset_path=f'datasets/{config["dataset"]}',
    per_device_train_batch_size=config['batch_size'],
    model_name=config['model_name'],
    output_dir=f'output/{config["model_name"].replace("/", "-")}-{output_extra_detail}-{config["dataset"]}',
    use_4bit=False,
    use_8bit=False,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_accumulation_steps=config['gradient_steps'],
    llm_int8_enable_fp32_cpu_offload=True,
    per_device_eval_batch_size=4,
    num_train_epochs=config['epochs'],
    max_seq_length=1024,
    save_steps=1000,
    load_in_4bit=False,
    lora_r=config['lora_r']
)

print(f"Output directory: {settings.output_dir}")

Output directory: output/unsloth-gemma-3-4b-it-lora-r16-qkvgud-bs8-ac1-e5-fp-aae2/adus


## Find Model Directories

Determine which model snapshots to run inference on

In [5]:
dirs = []
if config['all_snapshots']:
    dirs = [os.path.join(settings.output_dir, d) for d in os.listdir(settings.output_dir) if os.path.isdir(os.path.join(settings.output_dir, d)) and 'runs' not in d and 'logs' not in d]
    dirs = sorted(dirs, key=lambda x: int(x.split("-")[-1]))
else:
    dirs = [settings.output_dir]

print(f'Will run inference on {len(dirs)} model(s)')
for i, d in enumerate(dirs):
    print(f'  {i+1}. {d}')

Will run inference on 5 model(s)
  1. output/unsloth-gemma-3-4b-it-lora-r16-qkvgud-bs8-ac1-e5-fp-aae2/adus/checkpoint-669
  2. output/unsloth-gemma-3-4b-it-lora-r16-qkvgud-bs8-ac1-e5-fp-aae2/adus/checkpoint-1338
  3. output/unsloth-gemma-3-4b-it-lora-r16-qkvgud-bs8-ac1-e5-fp-aae2/adus/checkpoint-2007
  4. output/unsloth-gemma-3-4b-it-lora-r16-qkvgud-bs8-ac1-e5-fp-aae2/adus/checkpoint-2676
  5. output/unsloth-gemma-3-4b-it-lora-r16-qkvgud-bs8-ac1-e5-fp-aae2/adus/checkpoint-3345


## Run Inference

Load each model and generate predictions for the test dataset

In [6]:
# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=dirs[0],
    max_seq_length=settings.max_seq_length,
    dtype=settings.dtype,
    load_in_4bit=settings.load_in_4bit,
)
FastLanguageModel.for_inference(model)
# Configure tokenizer
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
if 'tiny' in settings.model_name.lower():
    tokenizer.chat_template = chat_templates.tiny_llama
elif 'llama-2' in settings.model_name.lower() or 'mistral' in settings.model_name.lower() or 'zephyr' in settings.model_name.lower():
    tokenizer.chat_template = chat_templates.llama_2
elif 'phi' in settings.model_name.lower():
    tokenizer.chat_template = chat_templates.phi2
# Setup prediction directory
pred_dir = settings.output_dir.replace('output', 'preds')
if config['all_snapshots']:
    pred_dir = re.sub('-e\\d+', f'-e{i+1:02d}', pred_dir)
os.makedirs(pred_dir, exist_ok=True)
print(f'Prediction directory: {pred_dir}')

# GPU memory usage
gpu_usage = get_gpu_memory_usage()
print(f'GPU usage of this model: {gpu_usage} MB\n')

==((====))==  Unsloth 2025.12.1: Fast Gemma3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.57 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Prediction directory: preds/unsloth-gemma-3-4b-it-lora-r16-qkvgud-bs8-ac1-e05-fp-aae2/adus
GPU usage of this model: 8352.60205078125 MB



In [19]:
import traceback
from transformers import TextStreamer

# Process test files
test_path = settings.dataset_path + "/test"
for root, _, files in os.walk(test_path):
    for f in tqdm(files, desc="Processing test files"):
        try:
            with open(os.path.join(root, f)) as f1, torch.no_grad():
                sample = convert_to_chat_json(f1.read(), config['remove_system_message'], is_gemma_3='gemma-3' in config['model_name'].lower())
                prompt = []
                response_length = 0
                for item in sample:
                    if item['role'] != 'assistant':
                        prompt.append(item)
                    else:
                        response_length = int(len(item['content'][0]['text']) * 1.5)
                        print(response_length)
                inputs = tokenizer.apply_chat_template(
                    prompt, 
                    add_generation_prompt = True, # Must add for generation
                    tokenize = True,
                    return_tensors = "pt",
                    return_dict = True,
                ).to('cuda')
                
                # Get the prompt length to skip it later
                prompt_length = inputs['input_ids'].shape[1]
                output = model.generate(**inputs, max_new_tokens=response_length, streamer=TextStreamer(tokenizer, skip_prompt=True))
                # Decode only the generated tokens (skip the prompt)
                response = tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True)
                # Save prediction
                with open(f"{pred_dir}/{f.replace('.json', '')}.txt", 'w') as f2:
                    f2.write(response)
                    
        except Exception as e:
            print(traceback)
            traceback.print_exc()
            # print(f"\nError processing {f}:")
            # print(f"  Error: {e}")
            # print(f"  Prompt: {prompt}")
        break

Processing test files:   0%|          | 0/1368 [00:00<?, ?it/s]

294
<Premise>Via internet they can have connection with the other children all around the world, therefor they can exchange their knowledge, they can learn different cultures</Premise><end_of_turn>


Processing test files:   0%|          | 0/1368 [00:04<?, ?it/s]

<Premise>Via internet they can have connection with the other children all around the world, therefor they can exchange their knowledge, they can learn different cultures</Premise>


In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()